In [24]:
def add_one(number: int):
    return number + 1

add_one(number="hello") # Crashes later with a messy error

TypeError: can only concatenate str (not "int") to str

In Python, you can "hint" at what type of data a function should get. For example:
def save_json(path: Path, data: dict):
Usually, Python ignores these hints. You could pass a number instead of a dictionary, and Python wouldn't complain until the code actually crashes deep inside the function.

`@ensure_annotations is a decorator that forces Python to check these hints while the code is running.`

    1.Stop Bugs Instantly
    2.Clearer Error
    3.Better Teamwork

In [15]:
from ensure import ensure_annotations

@ensure_annotations
def add_numbers(a: int, b: int) -> int:
    return a + b

# This works fine
add_numbers(10, 20) 


30

In [16]:
# This will raise an "EnsureError" immediately because "20" is a string
add_numbers(10, "20")

EnsureError: Argument b of type <class 'str'> to <function add_numbers at 0x7f006c670a40> does not match annotation type <class 'int'>

In standard Python, if you have a dictionary, you have to access the data using square brackets and quotes:
data["database"]["port"]

`ConfigBox is a wrapper that allows you to access dictionary keys as if they were attributes (using a dot):`

    data.database.port

Standard: print(config['artifacts']['root_dir'])

ConfigBox: print(config.artifacts.root_dir)

``` bash
Feature         Standard Python Dictionary,     With ConfigBox
AccessStyle,    d['key'],                       d.key
Syntax,         Cluttered (quotes/brackets),    Clean (dot notation)
SpeedtoType     Slower,                         Faster

In [17]:
from box import ConfigBox

# A normal dictionary
d = {"name": "Xray-AI", "version": 3}

# Convert it
d2 = ConfigBox(d)

# Now you can use dots!
print(d2.name)    # XrayAI
print(d2.version) # Output: 3

Xray-AI
3


# Segmentation Loss Functions (Dice Loss + Focal Loss)

This document explains the implementation of **Dice Coefficient, Dice Loss, Focal Loss, and Hybrid Loss (Dice + Focal)** used in **Deep Learning Image Segmentation tasks**.

These losses are widely used in:

* Medical Image Segmentation
* Autonomous Driving
* Satellite Image Analysis
* Industrial Defect Detection

---

# 1. Dice Coefficient

## Concept

Dice Coefficient measures the **overlap between the ground truth mask and the predicted mask**.

Range:

```
0 → No overlap
1 → Perfect overlap
```

## Mathematical Formula

Dice coefficient:

$$
Dice = \frac{2 \times |A \cap B|}{|A| + |B|}
$$

Where:

* A = Ground truth mask
* B = Predicted mask
* |A ∩ B| = Intersection between prediction and ground truth

Expanded version:

$$
Dice = \frac{2 \sum (y_{true} * y_{pred})}{\sum y_{true} + \sum y_{pred}}
$$

---

# Dice Coefficient Code

```python
def dice_coef(y_true, y_pred, smooth=1.0):
```

Function that calculates Dice score.

Parameters:

* y_true → Ground truth mask
* y_pred → Predicted mask
* smooth → Prevents division by zero

---

## Convert to Tensor

```python
y_true_f = ops.convert_to_tensor(y_true, dtype="float32")
y_pred_f = ops.convert_to_tensor(y_pred, dtype="float32")
```

Why?

Deep learning frameworks operate on **tensors**, so the input arrays must be converted.

---

## Flatten the masks

```python
y_true_f = ops.reshape(y_true_f, [-1])
y_pred_f = ops.reshape(y_pred_f, [-1])
```

Example:

Original mask size

```
256 x 256
```

Flattened

```
65536 pixels
```

Flattening makes overlap calculation easier.

---

## Compute Intersection

```python
intersection = ops.sum(y_true_f * y_pred_f)
```

Explanation:

If

```
y_true = 1
y_pred = 1
```

Then that pixel contributes to the **intersection**.

---

## Dice Formula Implementation

```python
return (2. * intersection + smooth) / (ops.sum(y_true_f) + ops.sum(y_pred_f) + smooth)
```

This applies the Dice formula.

`smooth` prevents division errors when both masks are empty.

---

# 2. Dice Loss

Neural networks **minimize loss**, but Dice is a **score**.

So we convert it to loss:

$$
DiceLoss = 1 - Dice
$$

---

## Dice Loss Code

```python
def dice_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)
```

Example:

```
Dice score = 0.90
Loss = 0.10
```

---

# 3. Focal Loss

## Problem: Class Imbalance

In segmentation datasets:

Example (Medical image):

```
Background = 95%
Lesion area = 5%
```

Normal Binary Cross Entropy focuses too much on background.

---

# Focal Loss Solution

Focal Loss focuses on **hard examples**.

Mathematical Formula:

$$
FL(p_t) = -\alpha (1 - p_t)^{\gamma} log(p_t)
$$

Where

* α = class balancing parameter
* γ = focusing parameter
* p_t = model confidence

---

# Focal Loss Code

```python
def focal_loss(gamma=2.0, alpha=0.25):
```

Default parameters:

```
gamma = 2
alpha = 0.25
```

---

## Convert Ground Truth

```python
y_true = ops.convert_to_tensor(y_true, dtype="float32")
```

Convert labels to tensor.

---

## Prevent log(0)

```python
y_pred = ops.clip(y_pred, epsilon, 1.0 - epsilon)
```

If prediction becomes 0 or 1 exactly:

```
log(0) = undefined
```

Clipping prevents numerical errors.

---

## Binary Cross Entropy

```python
bce = - (y_true * ops.log(y_pred) + (1.0 - y_true) * ops.log(1.0 - y_pred))
```

Standard BCE formula:

$$
BCE = -(y log(p) + (1-y) log(1-p))
$$

---

## Compute Model Confidence

```python
p_t = (y_true * y_pred) + ((1 - y_true) * (1 - y_pred))
```

Meaning:

```
p_t = probability of correct prediction
```

---

## Focal Weight

```python
focal_weight = ops.power(1.0 - p_t, gamma)
```

Effect:

Easy examples → small weight

Hard examples → large weight

---

## Final Focal Loss

```python
loss = alpha * focal_weight * bce
```

This emphasizes difficult pixels.

---

## Mean Loss

```python
return ops.mean(loss)
```

Average loss across all pixels.

---

# 4. Hybrid Loss (Dice + Focal)

Combine both losses.

```python
def total_loss(y_true, y_pred):
    return focal_loss()(y_true, y_pred) + dice_loss(y_true, y_pred)
```

Why combine?

| Loss       | Purpose                       |
| ---------- | ----------------------------- |
| Dice Loss  | Measures segmentation overlap |
| Focal Loss | Handles class imbalance       |

Together they improve segmentation accuracy.

---

# When To Use These Loss Functions

Use in **binary segmentation problems**.

Examples:

```
Tumor segmentation
Lung segmentation
Road segmentation
Crack detection
Flood detection
```

---

# Industry Use Cases

## 1 Medical Imaging

Models:

* U-Net
* Attention U-Net
* DeepLab

Applications:

```
Brain tumor detection
Lung infection detection
Cancer segmentation
```

---

# 2 Autonomous Driving

Segmentation classes:

```
Road
Car
Pedestrian
Traffic sign
```

---

# 3 Satellite Imagery

Applications:

```
Building detection
Flood mapping
Crop segmentation
```

---

# 4 Industrial Inspection

Examples:

```
Metal crack detection
Surface defects
Manufacturing quality control
```

---

# Example Model Compile

```python
model.compile(
    optimizer="adam",
    loss=total_loss,
    metrics=[dice_coef]
)
```

This trains the segmentation model using **Hybrid Loss**.

---

# Interview Explanation (Important)

Question:

Why combine Dice Loss with Focal Loss?

Good Answer:

```
Dice Loss optimizes overlap between predicted and ground truth masks,
while Focal Loss focuses on hard-to-classify pixels and handles class imbalance.
Combining both improves segmentation performance especially for small objects.
```

---

# Summary

Dice Loss + Focal Loss is widely used in **real-world segmentation systems** because it:

* Handles class imbalance
* Focuses on difficult pixels
* Improves mask overlap
* Works well for small object detection

---

End of Documentation


In [18]:
# Intersection Calculation
# y_true_f  *  y_pred_f

# [1,0,1,1] * [1,0,0,1]
# = [1,0,0,1]

# Now sum:
# intersection = 1 + 0 + 0 + 1 = 2

# Dice Calculation
# Dice = (2 * intersection) / (sum(y_true) + sum(y_pred))

# sum(y_true) = 3
# sum(y_pred) = 2

# Dice = (2*2)/(3+2)
#      = 4/5
#      = 0.8


# Code Explanation

# flatten করা হচ্ছে: # y_true_f = ops.reshape(y_true_f, [-1])

# -1 মানে: # Make it a single long vector

# Example:

# Before reshape 
# (256,256)

# After reshape
# (65536,)

# কারণ

# 256 × 256 = 65536 pixels


In [19]:
import numpy as np

mask = np.array([
[1,0,1],
[0,1,0],
[1,1,0]
])

print("Original shape:", mask.shape)

flat = mask.reshape(-1)

print("Flattened:", flat)
print("Shape:", flat.shape)

Original shape: (3, 3)
Flattened: [1 0 1 0 1 0 1 1 0]
Shape: (9,)


কেন ops ব্যবহার করা হয়?

আগে Keras এ backend dependency ছিল:

    TensorFlow
    PyTorch
    JAX

`keras.ops ব্যবহার করলে একই code সব backend এ কাজ করতে পারে।`

Example:

    Backend	Works
    TensorFlow	✅
    PyTorch	✅
    JAX	✅

এটা বলে backend-agnostic code।

Tensor Dimensions

| Data Type       | Example       | Dimension |
| --------------- | ------------- | --------- |
| Scalar          | 5             | 0D        |
| Vector          | [1,2,3]       | 1D        |
| Matrix          | [[1,2],[3,4]] | 2D        |
| Image           | 224×224×3     | 3D        |
| Batch of images | 32×224×224×3  | 4D        |


Why Tensor Use In Deep Learning?

Because tensors support:

| Feature               | Why Important   |
| --------------------- | --------------- |
| GPU computation       | Faster training |
| Parallel computation  | Large datasets  |
| Auto differentiation  | Backpropagation |
| High dimensional data | Images, video   |


Real ML Example

Image segmentation training:

    Input image

    Shape:
    (256, 256, 3)

    After batch:

    (16, 256, 256, 3)

    Meaning

    16 images per batch

In [20]:
import numpy as np

x = np.array([1,2,3])
print(x)

[1 2 3]


In [21]:
import tensorflow as tf

x = tf.constant([1,2,3])
print(x)

tf.Tensor([1 2 3], shape=(3,), dtype=int32)


In [22]:
import torch

x = torch.tensor([1,2,3])
print(x)

tensor([1, 2, 3])


In [23]:
from keras import ops

x = ops.array([[1,2],[3,4]])
print(x)

tf.Tensor(
[[1 2]
 [3 4]], shape=(2, 2), dtype=int32)


# Dice Loss vs IoU for Image Segmentation

This document explains **Dice Coefficient**, **Dice Loss**, and **Intersection over Union (IoU)** used in **Image Segmentation models** such as **U-Net, Mask R-CNN, DeepLab**.

---

# 1. What is Image Segmentation?

Image segmentation means **classifying every pixel of an image**.

Example:

| Input Image | Ground Truth Mask |
|-------------|------------------|
| X-ray image | Lung mask |
| Road image | Lane mask |
| Medical scan | Tumor mask |

Each pixel has a label:

```
0 = background
1 = object
```

---

# 2. Dice Coefficient

Dice coefficient measures **similarity between predicted mask and ground truth mask**.

## Formula

$$
Dice = \frac{2|A \cap B|}{|A| + |B|}
$$

Where

- **A** = Ground Truth mask  
- **B** = Predicted mask  
- **A ∩ B** = overlapping pixels

---

# 3. Dice Loss

Dice loss is used during **model training**.

## Formula

$$
Dice\ Loss = 1 - Dice
$$

If prediction is perfect:

```
Dice = 1
Loss = 0
```

---

# 4. Dice Loss Python Implementation

```python
import tensorflow as tf
from keras import ops

def dice_loss(y_true, y_pred, smooth=1e-6):

    # Flatten masks
    y_true_f = ops.reshape(y_true, [-1])
    y_pred_f = ops.reshape(y_pred, [-1])

    # Calculate intersection
    intersection = ops.sum(y_true_f * y_pred_f)

    # Dice formula
    dice = (2 * intersection + smooth) / (
        ops.sum(y_true_f) + ops.sum(y_pred_f) + smooth
    )

    return 1 - dice
```

---

# 5. Line-by-Line Code Explanation

## Step 1: Flatten the masks

```python
y_true_f = ops.reshape(y_true, [-1])
y_pred_f = ops.reshape(y_pred, [-1])
```

Original mask:

```
256 × 256
```

Flattened mask:

```
65536 pixels
```

Example:

Before flatten

```
[[1,0,1],
 [0,1,1]]
```

After flatten

```
[1,0,1,0,1,1]
```

Flattening converts a **2D mask into a 1D vector** so pixel comparison becomes easier.

---

## Step 2: Calculate intersection

```python
intersection = ops.sum(y_true_f * y_pred_f)
```

Example:

| Ground Truth | Prediction |
|--------------|------------|
| 1 | 1 |
| 0 | 1 |
| 1 | 1 |

Only `(1 × 1)` contributes to overlap.

---

# 6. Intersection over Union (IoU)

IoU measures how much **prediction overlaps with ground truth**.

## Formula

$$
IoU = \frac{|A \cap B|}{|A \cup B|}
$$

Where

- **Intersection** = common pixels
- **Union** = total pixels from both masks

---

# 7. IoU Python Implementation

```python
def iou_score(y_true, y_pred, smooth=1e-6):

    y_true_f = ops.reshape(y_true, [-1])
    y_pred_f = ops.reshape(y_pred, [-1])

    intersection = ops.sum(y_true_f * y_pred_f)

    union = ops.sum(y_true_f) + ops.sum(y_pred_f) - intersection

    return (intersection + smooth) / (union + smooth)
```

---

# 8. Dice vs IoU Difference

$$
| Metric | Formula | Best For |
$$
$$
|--------|---------|----------|
$$

$$
| Dice   | \frac{2 |A \cap B|}{|A| + |B|} | Small objects |
$$

$$
| IoU    | \frac{|A \cap B|}{|A \cup B|} | Benchmark evaluation |
$$
Relationship between Dice and IoU:

$$
Dice = \frac{2 \times IoU}{1 + IoU}
$$

---

# 9. Example Calculation

Suppose:

```
Ground Truth pixels = 100
Predicted pixels = 120
Overlap pixels = 80
```

### Dice

$$
Dice = \frac{2 × 80}{100 + 120} = 0.72
$$

### IoU

$$
IoU = \frac{80}{100 + 120 - 80} = 0.57
$$

---

# 10. Why Dice is Better than Accuracy

In segmentation, most pixels are **background**.

Example:

```
Background = 95%
Object = 5%
```

If model predicts **all background**:

```
Accuracy = 95%
```

But the model **detected nothing**.

Dice measures **true overlap**, so it is more reliable.

---

# 11. Example Model Training

```python
model.compile(
    optimizer="adam",
    loss=dice_loss,
    metrics=["accuracy"]
)
```

---

# 12. Real World Applications

### Medical Imaging

- Tumor segmentation
- Lung segmentation
- Brain MRI analysis

### Autonomous Driving

- Road segmentation
- Lane detection
- Pedestrian detection

### Satellite Vision

- Flood mapping
- Building detection
- Forest segmentation

---

# Summary

| Metric | Usage |
|------|------|
Dice | Training segmentation models |
IoU | Evaluation metric |

Both are widely used in **Computer Vision and Medical AI**.